# EduGuard-AI — Real OULAD dataset in Google Colab
This notebook clones the GitHub project, downloads the original public OULAD dataset from UCI, trains the two AI models, evaluates them on held-out records, and launches the Streamlit dashboard. No Drive setup or manual CSV uploads needed.

**Note:** results are preliminary research estimates, not validated intervention outcomes.

In [ ]:
!git clone https://github.com/cseteesta-bit/EduGuard-AI.git
%cd EduGuard-AI

In [ ]:
%pip -q install -r requirements.txt

## Download the official OULAD dataset
The source is UCI Machine Learning Repository dataset 349 (CC BY 4.0). The archive is approximately 45 MB; the extracted VLE CSV is much larger. Colab disk space is required.

In [ ]:
from pathlib import Path
import urllib.request, zipfile, shutil
root=Path("data/oulad")
root.mkdir(parents=True,exist_ok=True)
required=["studentInfo.csv","studentVle.csv","studentAssessment.csv","assessments.csv"]
url="https://archive.ics.uci.edu/static/public/349/open%2Buniversity%2Blearning%2Banalytics%2Bdataset.zip"
archive=Path("/content/oulad_original.zip")
if not all((root/f).exists() for f in required):
    print("Downloading official OULAD dataset from UCI...")
    urllib.request.urlretrieve(url,archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall(root)
    # Account for datasets packed inside a nested folder or an inner ZIP.
    nested=list(root.rglob("*.zip"))
    for zpath in nested:
        with zipfile.ZipFile(zpath) as z:
            z.extractall(zpath.parent)
    for name in required:
        if (root/name).exists(): continue
        matches=[p for p in root.rglob(name) if p.is_file()]
        if not matches: raise FileNotFoundError(f"{name} not found after extraction")
        shutil.copy2(matches[0],root/name)
    archive.unlink(missing_ok=True)
print({name:round((root/name).stat().st_size/1e6,1) for name in required})

## Train models and inspect real-data metrics
The project selects between Logistic Regression and Random Forest using validation F1 and calibrates conformal prediction sets using a separate calibration partition.

In [ ]:
import subprocess, json
from pathlib import Path
subprocess.run(['python','train.py','--data-dir','data/oulad'],check=True)
metrics=json.loads(Path('models/metrics.json').read_text())
print('Selected model:',metrics['selected_model'])
for model,details in metrics['models'].items():
    print(model,{k:round(v,4) if isinstance(v,float) else v for k,v in details['test'].items() if k in ['accuracy','f1','roc_auc','coverage','avg_set_size']})

## Launch Streamlit dashboard

In [ ]:
import subprocess, time
from google.colab import output

# Colab's reverse proxy uses a different Host/Origin pair for WebSockets.
# This setting is ONLY for an isolated, temporary Colab demo.
subprocess.run(["pkill", "-f", "[s]treamlit run app.py"], check=False)
server = subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.address", "0.0.0.0",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
], stdout=open("/tmp/eduguard_streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
if server.poll() is not None:
    print(open("/tmp/eduguard_streamlit.log").read()[-3000:])
    raise RuntimeError("Streamlit server exited")
print("Open dashboard:", output.eval_js("google.colab.kernel.proxyPort(8501)"))
display(output.serve_kernel_port_as_iframe(8501, height=850))


## Optional synthetic demonstration
Use only if the real dataset download is unavailable. Running this overwrites the trained real-data model until you rerun the real training cell.

In [ ]:
# !python train.py --demo